In [1]:
from dolfinx import io, fem, mesh as msh
from Training_utils import test_set
from SPDE_problems import *
from FEniCSx_solver import plot_fn
import torch
import torch_geometric as tg
from SPDE_problems import Data_to_solver
from SUPG_prediction_models import *
from FEniCSx_solver import interpolate_expr


mlp, gcn, sage, gat, gatv2 = MLP(), GCN(), SAGE(), GAT(), GATv2()
self_supervised_mlp, self_supervised_gcn, self_supervised_sage, self_supervised_gat, self_supervised_gatv2 = MLP(), GCN(), SAGE(), GAT(), GATv2()
sigmoid_gatv2, sigmoid_t_gatv2, sigmoid_q_gatv2, sigmoid_Icross_gatv2, sigmoid_Il2_gatv2 = SigmoidRestriction(GATv2), SigmoidRestriction(GATv2), SigmoidRestriction(GATv2), SigmoidRestriction(GATv2), SigmoidRestriction(GATv2)
clamp_gatv2 = ClampRestriction(GATv2)

mlp_state = torch.load("data/models/MLP.pth")
mlp.load_state_dict(mlp_state['model_state'])
gcn_state = torch.load("data/models/GCN.pth")
gcn.load_state_dict(gcn_state['model_state'])
sage_state = torch.load("data/models/SAGE.pth")
sage.load_state_dict(sage_state['model_state'])
gat_state = torch.load("data/models/GAT.pth")
gat.load_state_dict(gat_state['model_state'])
gatv2_state = torch.load("data/models/GATv2.pth")
gatv2.load_state_dict(gatv2_state['model_state'])

self_supervised_mlp_state = torch.load("data/models/MLP_self_supervised.pth")
self_supervised_mlp.load_state_dict(self_supervised_mlp_state['model_state'])
self_supervised_gcn_state = torch.load("data/models/GCN_self_supervised.pth")
self_supervised_gcn.load_state_dict(self_supervised_gcn_state['model_state'])
self_supervised_sage_state = torch.load("data/models/SAGE_self_supervised.pth")
self_supervised_sage.load_state_dict(self_supervised_sage_state['model_state'])
self_supervised_gat_state = torch.load("data/models/GAT_self_supervised.pth")
self_supervised_gat.load_state_dict(self_supervised_gat_state['model_state'])
self_supervised_gatv2_state = torch.load("data/models/GATv2_self_supervised.pth")
self_supervised_gatv2.load_state_dict(self_supervised_gatv2_state['model_state'])

sigmoid_gatv2_state = torch.load("data/models/sigmoid_GATv2.pth")
sigmoid_gatv2.load_state_dict(sigmoid_gatv2_state['model_state'])
sigmoid_t_gatv2_state = torch.load("data/models/sigmoid_t_GATv2.pth")
sigmoid_t_gatv2.load_state_dict(sigmoid_t_gatv2_state['model_state'])
sigmoid_q_gatv2_state = torch.load("data/models/sigmoid_q_GATv2.pth")
sigmoid_q_gatv2.load_state_dict(sigmoid_q_gatv2_state['model_state'])
sigmoid_Icross_gatv2_state = torch.load("data/models/sigmoid_Icross_GATv2.pth")
sigmoid_Icross_gatv2.load_state_dict(sigmoid_Icross_gatv2_state['model_state'])
sigmoid_Il2_gatv2_state = torch.load("data/models/sigmoid_Il2_GATv2.pth")
sigmoid_Il2_gatv2.load_state_dict(sigmoid_Il2_gatv2_state['model_state'])

clamp_gatv2_state = torch.load("data/models/clamp_GATv2.pth")
clamp_gatv2.load_state_dict(clamp_gatv2_state['model_state'])

models = [
    'GAT', 
    'GAT_self_supervised', 
    'GATv2', 
    'GATv2_clamp_restricted', 
    'GATv2_self_supervised',
    'GATv2_sigmoid_restricted',
    'GATv2_sigmoid_restricted_Icross',
    'GATv2_sigmoid_restricted_Il2',
    'GATv2_sigmoid_restricted_quadrilateral',
    'GATv2_sigmoid_restricted_triangular',
    'GCN',
    'GCN_self_supervised',
    'MLP',
    'MLP_self_supervised',
    'SAGE',
    'SAGE_self_supervised']
nets = [
    gat,
    self_supervised_gat,
    gatv2,
    clamp_gatv2,
    self_supervised_gatv2,
    sigmoid_gatv2,
    sigmoid_Icross_gatv2,
    sigmoid_Il2_gatv2,
    sigmoid_q_gatv2,
    sigmoid_t_gatv2,
    gcn,
    self_supervised_gcn,
    mlp,
    self_supervised_mlp,
    sage,
    self_supervised_sage
]
len(nets)

16

In [1]:
import pyvista as pv
from FEniCSx_solver import fem_plotter_grid

from SPDE_problems import *

for num in range(88):

    pv.global_theme.cmap = "coolwarm"
    fs , G = Data_to_solver(num, train = True)
    p = pv.Plotter(shape=(2,2))
    p.subplot(0,0)
    p.add_text(
        "u_std",
        position="upper_left",
        font_size=12,
        color="black"
    )
    weights = G.y
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.uh)
    p.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)

    p.show_grid(font_size = 8)
    p.subplot(0,1)
    p.add_text(
        "u_opt",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    fs.set_weights(weights.view(-1).detach().numpy())
    grid.add_data(fs.uh)
    p.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)
    print(num)
    p.show_grid(font_size = 8)
    p.subplot(1,0)
    p.add_text(
        "yh_opt",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.yh, point=False)
    p.add_mesh(grid.grid,
        scalar_bar_args={
            "title": ""
        })
    p.camera_position = 'xy'

    p.show_grid(font_size = 8)

    p.subplot(1,1)
    p.add_text(
        "yh_opt warped",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.yh)
    p.add_mesh(grid.grid.warp_by_scalar(factor=0.5), show_edges=True,
        scalar_bar_args={
            "title": "",
        })
    p.show_grid(font_size = 8)


    p.export_html(f"gallery/spde_{num:03d}.html")
    pthumb = pv.Plotter()

    grid = fem_plotter_grid(fs.Wh)
    fs.set_weights(weights.view(-1).detach().numpy())
    grid.add_data(fs.uh)
    pthumb.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)
    pthumb.screenshot(f'gallery/thumbnails/thumbnail_spde_{num:03d}.png')

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


Context leak detected, CoreAnalytics returned false


33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


Context leak detected, CoreAnalytics returned false


50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65


Context leak detected, CoreAnalytics returned false


66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82


Context leak detected, CoreAnalytics returned false


83
84
85
86
87


In [3]:
import pyvista as pv
from FEniCSx_solver import fem_plotter_grid

from SPDE_problems import *

for num in range(14):

    pv.global_theme.cmap = "coolwarm"
    fs , G = Data_to_solver(num, train = False)
    p = pv.Plotter(shape=(2,2))
    p.subplot(0,0)
    p.add_text(
        "u_std",
        position="upper_left",
        font_size=12,
        color="black"
    )
    weights = G.y
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.uh)
    p.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)

    p.show_grid(font_size = 8)
    p.subplot(0,1)
    p.add_text(
        "u_opt",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    fs.set_weights(weights.view(-1).detach().numpy())
    grid.add_data(fs.uh)
    p.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)
    print(num)
    p.show_grid(font_size = 8)
    p.subplot(1,0)
    p.add_text(
        "yh_opt",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.yh, point=False)
    p.add_mesh(grid.grid,
        scalar_bar_args={
            "title": ""
        })
    p.camera_position = 'xy'

    p.show_grid(font_size = 8)

    p.subplot(1,1)
    p.add_text(
        "yh_opt warped",
        position="upper_left",
        font_size=12,
        color="black"
    )
    grid = fem_plotter_grid(fs.Wh)
    grid.add_data(fs.yh)
    p.add_mesh(grid.grid.warp_by_scalar(factor=0.5), show_edges=True,
        scalar_bar_args={
            "title": "",
        })
    p.show_grid(font_size = 8)


    p.export_html(f"gallery/test_spde_{num:03d}.html")
    pthumb = pv.Plotter()

    grid = fem_plotter_grid(fs.Wh)
    fs.set_weights(weights.view(-1).detach().numpy())
    grid.add_data(fs.uh)
    pthumb.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)
    pthumb.screenshot(f'gallery/thumbnails/thumbnail_test_spde_{num:03d}.png')

0
1
2
3
4
5
6
7
8
9
10
11


Context leak detected, CoreAnalytics returned false


12
13


In [ ]:
from pathlib import Path

gallery_dir = Path("gallery")
thumb_dir = Path("gallery/thumbnails")

cards = ""

for html_file in sorted(gallery_dir.glob("*.html")):
    name = html_file.stem
    cards += f"""
    <div class="card">
        <a href="gallery/{name}.html">
            <img src="gallery/thumbnails/thumbnail_{name}.png">
        </a>
        <p>{name}</p>
    </div>
    """

template = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>DOLFINx Function Gallery</title>
<style>
.grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, 200px);
    justify-content: center;
    gap: 20px;
}}
.card img {{
    width: 200px;
    height: 150px;
    object-fit: cover;  /* crops nicely */
    border-radius: 8px;
}}
</style>
</head>
<body>
<h1>DOLFINx Function Gallery</h1>
<div class="grid">
{cards}
</div>
</body>
</html>
"""

Path("index.html").write_text(template)

19323

: 